In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from keras.callbacks import ModelCheckpoint
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score

import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import seaborn as sns

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
subs = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [4]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
train.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


# Dataset Preprocessing


In [6]:
train=train.drop(columns="Id")
test=test.drop(columns="Id")

Let's see how many NaNs we have for each feature.

In [7]:
nan_count=100*train.isna().sum().sort_values(ascending=False)/train.shape[0]
fig=px.bar(x=nan_count.index,y=nan_count.values, labels={"y": "Nan ammount (%)","x": "Feature"})
fig.show()

We can remove the features with NaN>40%, while the others will be handled replacing NaN with the respective median value.

In [8]:
train=train.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence',"FireplaceQu"])
test=test.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence',"FireplaceQu"])

In [9]:
numeric_features=[ feature  for feature in train.columns if  train[feature].dtypes!="object" and feature!="SalePrice"]
categorical_features=[ feature  for feature in train.columns if  train[feature].dtypes=="object"]

Now we are going to remove the last NaN values with the median value of each feature.

In [10]:
#replacing train NaNs with modes
nans=train.isna().sum()
nans=nans[nans>0]
for feature in nans.index:
    train[feature] = train[feature].fillna(train[feature].mode()[0])
#replacing test NaNs with modes
nans=test.isna().sum()
nans=nans[nans>0]
for feature in nans.index:
    test[feature] = test[feature].fillna(test[feature].mode()[0])

One hot encoding the categorical feature of train and test set.

In [11]:
for feature in categorical_features:    
    #some string values are present only in one of the dataset, so it is needed an unique list of both dataset to avoid conflicts
    for num, value in enumerate(np.unique((list(train[feature].unique())+list(test[feature].unique())))):          
        train[feature+"_"+str(num)]=pd.Series(train[feature]==value,dtype="int")        
        test[feature+"_"+str(num)]=pd.Series(test[feature]==value,dtype="int")
    train=train.drop(columns=feature)
    test=test.drop(columns=feature)
    

In [12]:
train

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_5,SaleType_6,SaleType_7,SaleType_8,SaleCondition_0,SaleCondition_1,SaleCondition_2,SaleCondition_3,SaleCondition_4,SaleCondition_5
0,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,0,0,0,1,0,0,0,0,1,0
1,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,0,0,0,1,0,0,0,0,1,0
2,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,0,0,0,1,0,0,0,0,1,0
3,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,0,0,0,1,1,0,0,0,0,0
4,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,62.0,7917,6,5,1999,2000,0.0,0,0,...,0,0,0,1,0,0,0,0,1,0
1456,20,85.0,13175,6,6,1978,1988,119.0,790,163,...,0,0,0,1,0,0,0,0,1,0
1457,70,66.0,9042,7,9,1941,2006,0.0,275,0,...,0,0,0,1,0,0,0,0,1,0
1458,20,68.0,9717,5,6,1950,1996,0.0,49,1029,...,0,0,0,1,0,0,0,0,1,0


Standard transformation
of the train and test test (only numeric features).

In [13]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
train[numeric_features]=scaler.fit_transform(train[numeric_features])
test[numeric_features]=scaler.transform(test[numeric_features])

Before feeding the data to the Neural Network, we peforme an PCA dimensionality reduction to reduce the noise of the data and to ease the calculation of the neural net.

In [14]:
x_train=train.drop(columns="SalePrice")
y_train=train['SalePrice']
pca = PCA(n_components=train.shape[1]-1)
x_train=pca.fit_transform(x_train)
fig=go.Figure()
fig.add_traces(go.Bar(x=np.arange(train.shape[1]-1),y=np.cumsum(pca.explained_variance_ratio_),name="Cumulative Variance"))
#n_comp will be the number of components that explains the 95% of the data variance
n_comp=np.where(np.cumsum(pca.explained_variance_ratio_)>0.95)[0][0]
fig.add_traces(go.Scatter(x=np.arange(train.shape[1]-1),y=[0.95]*(train.shape[1]-1),name="Variance at 95%"))
fig.update_layout(title="How many components we need?",xaxis_title="Components",yaxis_title="Cumulative Variance", font=dict(
        family="Arial",
        size=18,
    ))
fig.show()
print("With n_components="+str(n_comp)+" we have the 95% of the data variance, so we will choose this value.")

With n_components=71 we have the 95% of the data variance, so we will choose this value.


In [15]:
pca = PCA(n_components=n_comp+50)
x_train=pca.fit_transform(train.drop(columns=["SalePrice"]))

# Model definition and training


In [16]:
model = tf.keras.Sequential([
      layers.Dense(2048, activation='relu'),
      layers.Dropout(0.5),
      layers.Dense(2048, activation='relu'),
      layers.Dropout(0.5),
      layers.Dense(1)
  ])
model.compile(loss='mean_squared_error',optimizer=tf.keras.optimizers.Adamax(1e-3))

In [17]:
history = model.fit(x_train,y_train,validation_split=0.1,verbose=0, epochs=300)

In [18]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=np.arange(300), y=history.history['loss'],mode='lines', name='Train Loss'))
fig.add_trace(go.Scatter(x=np.arange(300), y=history.history['val_loss'],mode='lines', name='Validation Loss',))
fig.update_layout(title="MAE loss on train and validation set",xaxis_title="Epoch", yaxis_title="Loss", font=dict(
        family="Arial",
        size=18,
    ))
fig.show()

# Model evaluation and submission


In [19]:
print("Validation loss:",history.history['val_loss'][-1])
print("Training loss:",history.history['loss'][-1])
print("Loss on entire train set:",mean_absolute_error(model.predict(x_train),y_train))
print("R2 score(Train):",r2_score(model.predict(x_train),y_train))

Validation loss: 611624960.0
Training loss: 393524768.0
Loss on entire train set: 11475.995986729453
R2 score(Train): 0.9409048239451289


In [20]:
sub_preds = model.predict(pca.transform(test))
subs["SalePrice"] = sub_preds
subs.to_csv("submission.csv", index = False)
print("Submission done!")

Submission done!
